# 3Dn per-residue structural alphabet → JSON

3Dn of Petti et al., *Bioinformatics* 2025 (btaf458). Same output format as `foldseek_3di_per_residue.json`.

In [ ]:
import os, sys, json, types, warnings
from pathlib import Path
import numpy as np
from Bio.PDB import PDBParser
from Bio.SeqUtils import seq1
warnings.filterwarnings("ignore")

# ---- paths ----
REPO    = Path("../../../external/3Dn")
REF     = REPO / "refData"
PDB_DIR = Path("../../../data/rBMRB/cluster_30p/structures_split_filtered/test")
OUT_JSON = "../../../data/analysis/Fig4/tdn_results/3dn_per_residue.json"

# ---- lightweight jax stub: 3Dn sequence generation needs only numpy; jax is used only for alignment (sw_affine) ----
if "jax" not in sys.modules:
    _jx = types.ModuleType("jax"); _jx.numpy = np
    _jx.jit = lambda f=None, **k: (f if callable(f) else (lambda g: g))
    sys.modules["jax"] = _jx; sys.modules["jax.numpy"] = np

sys.path.insert(0, str(REPO))
from utils_example import (generateBinList, allBinsNeighborsPerProt,
                           getOneHot, seqFromCentroids)

# ---- load the trained 3Dn matrices ----
tMtx    = np.load(REF / "transition_mtx.npy")   # (1001, 1001)
centers = np.load(REF / "MI_centers.npy")        # (20, 1000) cluster centroids
allBins = generateBinList()                      # 1000 spatial x SS bins

# cluster index 0-19 -> letters A-T (distinct from the amino-acid letters used by 3Di)
TDN_ALPHABET = "ABCDEFGHIJKLMNOPQRST"

pdb_files = sorted(PDB_DIR.glob("*.pdb"))
print(f"Found {len(pdb_files)} PDB files | centers={centers.shape} tMtx={tMtx.shape}")

In [ ]:
# ---- 3Dn sequence of one PDB (list of cluster indices) ----
def compute_3dn(pdb_path):
    nbrs = allBinsNeighborsPerProt(str(pdb_path))
    nHot = getOneHot(nbrs, allBins)
    bn   = np.einsum("ik,kj->ij", nHot, tMtx)[:, :-1]   # blurry neighborhood, n×1000
    seq  = seqFromCentroids(bn, centers)                # [(idx,), ...]
    return [int(t[0]) for t in seq]

# ---- (resnum, resname) in the same residue order as 3Dn ----
# utils_example.getAllPositions iterates model -> chain -> residue in list(chain);
# the same order is used here so residues line up.
parser = PDBParser(QUIET=True)
def residue_list(pdb_path):
    st = parser.get_structure("x", str(pdb_path))
    out = []
    for model in st:
        for chain in model:
            for residue in list(chain):
                rn3 = residue.get_resname()
                aa1 = seq1(rn3) if len(rn3) == 3 else "X"
                out.append((residue.id[1], aa1 if aa1 else "X"))
    return out

In [ ]:
# ---- batch: align residues into {stem: [{resnum, resname, tdn, tdn_idx}, ...]} ----
result = {}
failed = []
for k, f in enumerate(pdb_files):
    stem = f.stem
    try:
        idxs = compute_3dn(f)
        res  = residue_list(f)
        if len(idxs) != len(res):
            failed.append((stem, f"len 3dn={len(idxs)} res={len(res)}"))
            continue
        result[stem] = [
            {"resnum": rn, "resname": aa,
             "tdn": TDN_ALPHABET[i], "tdn_idx": i}
            for (rn, aa), i in zip(res, idxs)
        ]
    except Exception as e:
        failed.append((stem, repr(e)[:120]))
    if (k + 1) % 25 == 0:
        print(f"  processed {k+1}/{len(pdb_files)}")

print(f"\nDone: {len(result)} structures | failed: {len(failed)}")
for s, why in failed[:20]:
    print("   ", s, why)

In [ ]:
# ---- save JSON + quick check ----
with open(OUT_JSON, "w") as fh:
    json.dump(result, fh, indent=2)
print("Saved:", OUT_JSON, "| num structures:", len(result))

k = next(iter(result))
print(f"\nsample [{k}] first 5 residues:")
for r in result[k][:5]:
    print("  ", r)

# consistency with the 3Di json (same keys and residue count per structure)
tdi_path = Path("../../../data/analysis/Fig4/tdi_results/foldseek_3di_per_residue.json")
if tdi_path.exists():
    tdi = json.load(open(tdi_path))
    common = set(result) & set(tdi)
    print(f"\nkeys: 3dn={len(result)} 3di={len(tdi)} common={len(common)}")
    diff = [k for k in common if len(result[k]) != len(tdi[k])]
    print("per-structure residue-count mismatches vs 3Di:", len(diff))
    if diff:
        print("  e.g.", diff[:10])